In [1]:
import pandas as pd
import numpy as np

FILE = "D:/Tushar/Copy of Master Data _290102026 2 - Copy.xlsx"

df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()


In [2]:
df = df[df["Category"].isin(["Repeater", "Stranger"])].copy()


In [3]:
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]


In [4]:
grp = df.groupby("Child Part", sort=False)

parts = grp.agg({
    "effective_daily_demand": "sum",
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Cycle Time": "first",
    "Vertical Machines": lambda x: list(
        set(",".join(x.astype(str)).split(","))
    ),
    "Category": "first"
}).reset_index()

parts.rename(columns={
    "Child Part": "part",
    "effective_daily_demand": "daily_demand",
    "Inventory_25": "inventory",
    "Minimum Quantity": "min_qty",
    "Cycle Time": "cycle_time",
    "Vertical Machines": "machines"
}, inplace=True)


In [5]:
parts["net_required_qty"] = (
    parts["daily_demand"]
    + parts["min_qty"]
    - parts["inventory"]
).clip(lower=0)


In [6]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m.strip(),
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)


In [7]:
CHANGEOVER = 40
CAPACITY = 1320
TARGET_DAYS = 3

def compute_score(row):
    # Inventory pain
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    # Relief potential
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)
    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    # Production time
    prod_time = qty_if_made * row["cycle_time"]

    # Setup penalty
    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    # Monopoly penalty
    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)


In [8]:
print("dfm shape: ", dfm.shape)
print(dfm.head())

dfm shape:  (130, 7)
                 part  category machine  daily_demand  inventory  cycle_time  \
0  14SW030081-00001X0  Repeater  M.P-01           0.0        0.0        45.0   
1  14SW030082-00001X0  Stranger  M.P-05         210.0        0.0        45.0   
2  14SW110487-00002X0  Stranger  M.P-08          95.0        0.0        20.0   
3  14SW110487-00009X0  Stranger  M.P-17          95.0        0.0        24.0   
4  14SW220197-00005X0  Repeater  M.P-05        1040.0        0.0        25.0   

   net_required_qty  
0        895.258065  
1        689.709677  
2        285.258065  
3        285.258065  
4       3399.838710  


In [9]:
dfm["score"] = dfm.apply(compute_score, axis=1)


In [10]:
selected = []

for m, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(3))

selected = pd.concat(selected).reset_index(drop=True)


In [11]:
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]


In [12]:
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== DAILY PLAN (REPEATER + STRANGER ONLY) =====")
print(final_plan)



===== DAILY PLAN (REPEATER + STRANGER ONLY) =====
         machine                part  category  qty_today  prod_time_min  \
0         M.P-01       S12094-003A0X  Stranger         21         1008.0   
1         M.P-01       S12071-001A0X  Stranger         30         1050.0   
2         M.P-01  14SW030081-00001X0  Repeater          0            0.0   
3         M.P-03       S31868-005A0X  Stranger        113         4520.0   
4         M.P-03       S31842-004A0X  Stranger        206         6180.0   
5         M.P-03       S31841-008A0X  Repeater        903         8127.0   
6         M.P-04  14SW311101-00003X0  Repeater        300         6900.0   
7         M.P-04       S32073-012A0X  Stranger        675        10125.0   
8         M.P-04       S11398-017A0X  Repeater        933        13995.0   
9         M.P-05  14SW030082-00001X0  Stranger        630        28350.0   
10        M.P-05       S12095-004A0X  Repeater        879        39555.0   
11        M.P-05  14SW220197-00005X0 